# Step 2: Professional Data Cleaning & Splitting

Because of what we saw in the Histograms in the last notebook, we MUST clean this data so XGBoost can read it perfectly.
We will run the 4 steps: Missing Values, Outlier Capping, Standard Scaler, and Stratified Split.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

data = pd.read_csv(r"../data/sampled_uplift.csv")
feature_cols = [f'f{i}' for i in range(12)]

print("Raw Data Successfully Loaded!")

In [ ]:
print("--- 1. Handling Missing Values ---")
missing_count = data[feature_cols].isnull().sum().sum()
if missing_count > 0:
    data[feature_cols] = data[feature_cols].fillna(data[feature_cols].median())
    print(f"Filled {missing_count} blank spaces with the median.")
else:
    print("No missing values found.")

print("\n--- 2. Outlier Capping (Winsorization) ---")
for col in feature_cols:
    upper_limit = data[col].quantile(0.99)
    lower_limit = data[col].quantile(0.01)
    data[col] = np.clip(data[col], a_min=lower_limit, a_max=upper_limit)
print("Extreme outliers chopped off at the 99th percentile.")

print("\n--- 3. Feature Scaling (Standardization) ---")
scaler = StandardScaler()
data[feature_cols] = scaler.fit_transform(data[feature_cols])
print("Features mathematically squished to identical scales.")

### The Final Split
Now we cut the perfectly clean 500k rows into our 80% Training File and our 20% Evaluation File (Stratified).

In [ ]:
print("--- 4. Stratified Train / Test Split ---")
X = data[feature_cols + ['treatment']] # Input Data
y = data['conversion']                 # The Answer (Target)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.20,      
    random_state=42,    
    stratify=X['treatment'] # Forces exactly 85% Treatment / 15% Control
)

print(f"Training Rows: {X_train.shape[0]}")
print(f"Testing Rows:  {X_test.shape[0]}")

### Save the Cleaned Splits for AI Training!
Let's save these 4 mathematical arrays to our `data/` folder so we can load them seamlessly into XGBoost in the next phase.

In [ ]:
X_train.to_csv("../data/X_train.csv", index=False)
X_test.to_csv("../data/X_test.csv", index=False)
y_train.to_csv("../data/y_train.csv", index=False)
y_test.to_csv("../data/y_test.csv", index=False)

print("✅ SUCCESS! ALL CLEAN DATA SAVED!")
print("We are officially ready to build the XGBoost Model.")